# Conformer

Gulati et al. 2020, *"Conformer: Convolution-augmented Transformer for Speech Recognition"* ([arXiv:2005.08100](https://arxiv.org/abs/2005.08100)).

Each block is a macaron sandwich: half-FFN, bidirectional self-attention, a real conv module (pointwise -> GLU -> depthwise -> BatchNorm -> Swish -> pointwise), half-FFN, final LayerNorm. Trained here on real Google Speech Commands (core 10 spoken words), log-mel spectrogram features.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import torchaudio
import matplotlib.pyplot as plt

from transformer_playground.data import load_speech_commands
from transformer_playground.device import resolve_device
from model import ConformerModel

device = resolve_device('auto')
print('device:', device)

In [ ]:
(train_wavs, train_labels), (test_wavs, test_labels), words = load_speech_commands(seed=0)
print(f'real Speech Commands (core {len(words)} words): {len(train_wavs)} train, {len(test_wavs)} test clips')

n_mels = 40
mel = torchaudio.transforms.MelSpectrogram(sample_rate=16000, n_fft=400, hop_length=160, n_mels=n_mels)
to_db = torchaudio.transforms.AmplitudeToDB()

def to_features(wavs):
    return to_db(mel(wavs)).transpose(1, 2)

train_x = to_features(torch.from_numpy(train_wavs)).to(device)
train_y = torch.from_numpy(train_labels).to(device)
test_x = to_features(torch.from_numpy(test_wavs)).to(device)
test_y = torch.from_numpy(test_labels).to(device)
print('feature shape (T, n_mels):', tuple(train_x.shape[1:]))

In [ ]:
model = ConformerModel(n_mels=n_mels, num_classes=len(words), max_len=train_x.shape[1]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
batch_size = 32

def iterate_batches(x, y, batch_size, shuffle):
    n = x.shape[0]
    idx = torch.randperm(n) if shuffle else torch.arange(n)
    for i in range(0, n, batch_size):
        b = idx[i:i+batch_size]
        yield x[b], y[b]

@torch.no_grad()
def evaluate():
    model.eval()
    correct, total = 0, 0
    for xb, yb in iterate_batches(test_x, test_y, batch_size, shuffle=False):
        preds = model(xb).argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.numel()
    model.train()
    return correct / total

history = {'train_loss': [], 'test_acc': []}
for epoch in range(10):
    model.train()
    last_loss = None
    for xb, yb in iterate_batches(train_x, train_y, batch_size, shuffle=True):
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        last_loss = loss.item()
    acc = evaluate()
    history['train_loss'].append(last_loss)
    history['test_acc'].append(acc)
    print(f'epoch {epoch:3d} | train_loss {last_loss:.4f} | test_acc {acc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss']); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('train loss')
axes[1].plot(history['test_acc']); axes[1].set_xlabel('epoch'); axes[1].set_ylabel('test acc')
fig.suptitle('Conformer on real Google Speech Commands (core 10 words)')
plt.show()